# <font color="#418FDE" size="6.5" uppercase>**MLP und Faltungen**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Trainieren ein kleines Zwei-Schichten-MLP mit NumPy auf einfachen Klassifikationsdaten. 
- Vergleichen Mini-Batches, Shuffling, SGD, Momentum, Regularisierung und Early Stopping. 
- Implementieren grundlegende Faltungs- und Poolingoperationen und planen Tensorformen. 


## **1. MLP trainieren**

### **1.1. Kleine Datensätze**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_01_01.jpg?v=1784807937" width="250">



>* Kleine Daten machen MLP-Training anschaulich
>* Entscheidungsgrenzen zeigen gelernte Regeln

>* MLPs erkennen nichtlineare Muster in Daten
>* Kleine Datensätze zeigen Training und Überanpassung

>* Fehler und Entscheidungen gezielt untersuchen
>* Training als kontrolliertes Experiment verstehen



In [ ]:
#@title Python-Code - Kleine Datensätze

# Wir trainieren ein kleines MLP mit NumPy.
# Der Fokus liegt auf kleinen Klassifikationsdaten.
# Am Ende sehen wir Genauigkeit und Entscheidungsgrenze.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sklearn

# Ein kleiner Datensatz macht jeden Trainingsschritt überschaubar.
features, labels = make_moons(n_samples=240, noise=0.22, random_state=42)
labels = labels.reshape(-1, 1)

# Die Aufteilung prüft, ob das Netz neue Punkte einordnet.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.3, stratify=labels, random_state=42
)

# Skalierung wird nur mit Trainingsdaten gelernt.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Diese Prüfung schützt vor unerwarteten Formfehlern.
if X_train.shape[1] != 2 or y_train.shape[1] != 1:
    raise ValueError("Erwartet werden zwei Merkmale und ein Ziel.")

# Kleine zufällige Gewichte starten das Zwei-Schichten-MLP.
rng = np.random.default_rng(42)
hidden_units = 8
W1 = rng.normal(0, 0.5, size=(2, hidden_units))
b1 = np.zeros((1, hidden_units))
W2 = rng.normal(0, 0.5, size=(hidden_units, 1))
b2 = np.zeros((1, 1))

# Sigmoid wandelt die Ausgabe in Klassenwahrscheinlichkeiten um.
def sigmoid(values):
    return 1 / (1 + np.exp(-values))

# Das Training nutzt vollständige Batches für klare Nachvollziehbarkeit.
learning_rate = 0.08
epochs = 900
sample_count = X_train.shape[0]

# Vorwärtslauf, Fehler und Rückwärtslauf wiederholen sich.
for epoch in range(epochs):
    hidden_linear = X_train @ W1 + b1
    hidden = np.tanh(hidden_linear)
    output = sigmoid(hidden @ W2 + b2)

    output_error = output - y_train
    dW2 = hidden.T @ output_error / sample_count
    db2 = np.mean(output_error, axis=0, keepdims=True)

    hidden_error = (output_error @ W2.T) * (1 - hidden * hidden)
    dW1 = X_train.T @ hidden_error / sample_count
    db1 = np.mean(hidden_error, axis=0, keepdims=True)

    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1

# Eine kleine Hilfsfunktion berechnet Vorhersagen des Netzes.
def predict_proba(data):
    hidden = np.tanh(data @ W1 + b1)
    return sigmoid(hidden @ W2 + b2)

# Genauigkeit zeigt den Lernerfolg auf bekannten und neuen Daten.
train_pred = predict_proba(X_train) >= 0.5
test_pred = predict_proba(X_test) >= 0.5
train_accuracy = np.mean(train_pred == y_train)
test_accuracy = np.mean(test_pred == y_test)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Trainingsgenauigkeit: {train_accuracy:.2f}")
print(f"Testgenauigkeit: {test_accuracy:.2f}")

# Ein Gitter macht die gelernte Entscheidungsgrenze sichtbar.
x_min = X_train[:, 0].min() - 0.6
x_max = X_train[:, 0].max() + 0.6
y_min = X_train[:, 1].min() - 0.6
y_max = X_train[:, 1].max() + 0.6

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 160), np.linspace(y_min, y_max, 160)
)
grid = np.c_[xx.ravel(), yy.ravel()]
probabilities = predict_proba(grid).reshape(xx.shape)

# Die Grafik verbindet Datenpunkte mit der gelernten Grenze.
fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(xx, yy, probabilities, levels=20, cmap="RdBu", alpha=0.35)
ax.contour(xx, yy, probabilities, levels=[0.5], colors="black", linewidths=2)

ax.scatter(
    X_test[:, 0], X_test[:, 1], c=y_test.ravel(), cmap="RdBu", edgecolor="white"
)
ax.set_title("Zwei-Schichten-MLP auf kleinem Datensatz")
ax.set_xlabel("Merkmal 1, skaliert")
ax.set_ylabel("Merkmal 2, skaliert")
plt.show()



### **1.2. Netzarchitektur**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_01_02.jpg?v=1784807933" width="250">



>* Eingaben werden über verborgene Neuronen verarbeitet
>* Architektur bestimmt Einheiten und Klassenausgaben

>* Nichtlinearität ermöglicht flexible verborgene Detektoren
>* Ausgabegröße und Schichtgröße passend wählen

>* Tensorformen zeigen passende Schichtverbindungen.
>* Bias und Architektur führen zur Entscheidung.



In [ ]:
#@title Python-Code - Netzarchitektur

# Dieses Beispiel plant eine kleine MLP-Architektur.
# Tensorformen zeigen den Weg durch das Netz.
# Die Ausgabe bestätigt passende Matrixgrößen.

import numpy as np

# Vier Beispiele mit je zwei Eingabemerkmalen.
X = np.array([[0.2, 0.8], [0.9, 0.1], [0.4, 0.6], [0.7, 0.3]])
y = np.array([0, 1, 0, 1])

# Die Architektur legt jede Schichtgröße fest.
input_size = X.shape[1]
hidden_size = 5
output_size = 2

# Kleine Zufallsgewichte machen die Rechnung reproduzierbar.
rng = np.random.default_rng(42)
W1 = rng.normal(0.0, 0.1, size=(input_size, hidden_size))
b1 = np.zeros(hidden_size)

# Die zweite Schicht verbindet verborgen und Ausgabe.
W2 = rng.normal(0.0, 0.1, size=(hidden_size, output_size))
b2 = np.zeros(output_size)

# Diese Prüfung erkennt Formfehler früh.
if W1.shape != (input_size, hidden_size):
    raise ValueError("W1 passt nicht zur Eingabe und verborgenen Schicht.")

# Die verborgene Schicht erzeugt eine neue Darstellung.
hidden_linear = X @ W1 + b1
hidden_activation = np.maximum(0.0, hidden_linear)

# Die Ausgabeschicht erzeugt zwei Klassenscores.
scores = hidden_activation @ W2 + b2
predicted_class = np.argmax(scores, axis=1)

# Die Softmax-Funktion macht Scores vergleichbarer.
shifted_scores = scores - np.max(scores, axis=1, keepdims=True)
exp_scores = np.exp(shifted_scores)
probabilities = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

# Die Ausgabe zeigt Architektur und Datenfluss.
print(f"Eingabe X: {X.shape} = Beispiele x Merkmale")
print(f"W1: {W1.shape}, b1: {b1.shape} -> verborgen: {hidden_activation.shape}")
print(f"W2: {W2.shape}, b2: {b2.shape} -> Scores: {scores.shape}")
print(f"Wahre Klassen: {y.tolist()}")
print(f"Vorhersagen vor Training: {predicted_class.tolist()}")
print(f"Wahrscheinlichkeiten erstes Beispiel: {np.round(probabilities[0], 3).tolist()}")



### **1.3. Trainingsschleife verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_01_03.jpg?v=1784807935" width="250">



>* MLP lernt durch wiederholte Fehlerkorrektur
>* Schichten formen schrittweise bessere Klassengrenzen

>* Vorhersagen berechnen und Fehler messen
>* Gradienten bestimmen, Parameter mit Lernrate anpassen

>* Lernverlauf statt nur Endgenauigkeit beobachten
>* Verlustkurven zeigen Probleme und Fortschritt



In [ ]:
#@title Python-Code - Trainingsschleife verstehen

# Wir trainieren ein kleines MLP mit NumPy.
# Die Schleife zeigt Vorwärtslauf, Verlust und Update.
# Am Ende sieht man sinkenden Verlust und Genauigkeit.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sklearn

# Ein kleiner Datensatz macht die Trainingsschleife überschaubar.
features, labels = make_moons(n_samples=400, noise=0.22, random_state=42)
labels = labels.reshape(-1, 1)

# Die Aufteilung prüft, ob das Modell neue Punkte versteht.
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, stratify=labels, random_state=42
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Eine einfache Prüfung verhindert unklare Formfehler.
if X_train.shape[1] != 2 or y_train.shape[1] != 1:
    raise ValueError("Die Datenformen passen nicht zum Beispiel.")

# Kleine Zufallsgewichte starten das MLP reproduzierbar.
rng = np.random.default_rng(42)
hidden_units = 8
learning_rate = 0.08

W1 = rng.normal(0.0, 0.5, size=(2, hidden_units))
b1 = np.zeros((1, hidden_units))
W2 = rng.normal(0.0, 0.5, size=(hidden_units, 1))
b2 = np.zeros((1, 1))

# Diese Funktion wandelt Werte in Wahrscheinlichkeiten um.
def sigmoid(values):
    return 1.0 / (1.0 + np.exp(-values))

# Wir speichern wenige Messpunkte statt jede Epoche auszugeben.
epochs = 80
loss_history = []
checkpoints = []

# Jede Epoche enthält Vorwärtslauf, Rückwärtslauf und Update.
for epoch in range(epochs):
    hidden_linear = X_train @ W1 + b1
    hidden_activation = np.tanh(hidden_linear)
    output_linear = hidden_activation @ W2 + b2
    predictions = sigmoid(output_linear)

    clipped = np.clip(predictions, 1e-7, 1.0 - 1e-7)
    loss = -np.mean(y_train * np.log(clipped) + (1 - y_train) * np.log(1 - clipped))
    loss_history.append(loss)

    output_error = (predictions - y_train) / len(X_train)
    dW2 = hidden_activation.T @ output_error
    db2 = np.sum(output_error, axis=0, keepdims=True)

    hidden_error = (output_error @ W2.T) * (1.0 - hidden_activation ** 2)
    dW1 = X_train.T @ hidden_error
    db1 = np.sum(hidden_error, axis=0, keepdims=True)

    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1

    if epoch in [0, 19, 39, 79]:
        checkpoints.append((epoch + 1, loss))

# Die Testgenauigkeit nutzt Parameter nach der letzten Epoche.
test_hidden = np.tanh(X_test @ W1 + b1)
test_probabilities = sigmoid(test_hidden @ W2 + b2)
test_predictions = (test_probabilities >= 0.5).astype(int)
test_accuracy = np.mean(test_predictions == y_test)

print(f"scikit-learn Version: {sklearn.__version__}")
print("Verlust nach Epochen 1, 20, 40, 80:")
print([round(value, 3) for _, value in checkpoints])
print(f"Testgenauigkeit nach dem Training: {test_accuracy:.2f}")

# Die Kurve zeigt, ob die Updates den Fehler verringern.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, epochs + 1), loss_history, label="Trainingsverlust")
ax.set_title("Trainingsschleife eines kleinen Zwei-Schichten-MLP")
ax.set_xlabel("Epoche")
ax.set_ylabel("Binärer Kreuzentropie-Verlust")
ax.legend()
plt.show()



## **2. Training verbessern**

### **2.1. Mini Batches**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_02_01.jpg?v=1784807941" width="250">



>* Mini Batches teilen Daten in kleine Gruppen
>* Stabile Updates bei effizienter Hardware-Nutzung

>* Mini Batches schätzen Lernrichtungen aus Stichproben
>* Leichte Zufälligkeit stabilisiert und hilft beim Lernen

>* Batch-Größe steuert Tempo und Lernrauschen
>* Shuffling sorgt für ausgewogene Mini Batches



In [ ]:
#@title Python-Code - Mini Batches

# Dieses Beispiel zeigt Mini-Batches beim Training.
# Wir vergleichen kleine und große Batch-Größen.
# Die Verlustkurven zeigen unterschiedliche Lernsignale.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sklearn

# Wir erzeugen kleine Klassifikationsdaten ohne Download.
features, labels = make_classification(
    n_samples=600,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=42,
)

# Die Aufteilung bleibt durch random_state reproduzierbar.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42,
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Eine einfache Prüfung schützt vor Formfehlern.
if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("Trainingsdaten und Labels passen nicht zusammen.")

# Diese Funktion trainiert logistische Regression per Mini-Batch-SGD.
def train_with_batch_size(X, y, batch_size, epochs=35, learning_rate=0.15):
    rng = np.random.default_rng(42)
    weights = np.zeros(X.shape[1])
    bias = 0.0
    losses = []

    for epoch in range(epochs):
        order = rng.permutation(X.shape[0])
        X_shuffled = X[order]
        y_shuffled = y[order]

        for start in range(0, X.shape[0], batch_size):
            stop = start + batch_size
            X_batch = X_shuffled[start:stop]
            y_batch = y_shuffled[start:stop]

            logits = X_batch @ weights + bias
            predictions = 1.0 / (1.0 + np.exp(-logits))
            errors = predictions - y_batch

            weight_gradient = X_batch.T @ errors / X_batch.shape[0]
            bias_gradient = np.mean(errors)
            weights = weights - learning_rate * weight_gradient
            bias = bias - learning_rate * bias_gradient

        full_logits = X @ weights + bias
        full_predictions = 1.0 / (1.0 + np.exp(-full_logits))
        clipped = np.clip(full_predictions, 1e-7, 1 - 1e-7)
        loss = -np.mean(y * np.log(clipped) + (1 - y) * np.log(1 - clipped))
        losses.append(loss)

    return np.array(losses), weights, bias

# Kleine Batches aktualisieren häufiger und schwanken oft stärker.
small_losses, small_weights, small_bias = train_with_batch_size(
    X_train,
    y_train,
    batch_size=8,
)

# Große Batches liefern glattere, aber seltenere Aktualisierungen.
large_losses, large_weights, large_bias = train_with_batch_size(
    X_train,
    y_train,
    batch_size=128,
)

# Wir berechnen eine einfache Testgenauigkeit für beide Varianten.
def accuracy(X, y, weights, bias):
    probabilities = 1.0 / (1.0 + np.exp(-(X @ weights + bias)))
    predicted_labels = probabilities >= 0.5
    return np.mean(predicted_labels == y)

small_accuracy = accuracy(X_test, y_test, small_weights, small_bias)
large_accuracy = accuracy(X_test, y_test, large_weights, large_bias)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Batch 8: letzter Verlust {small_losses[-1]:.3f}, Testgenauigkeit {small_accuracy:.3f}")
print(f"Batch 128: letzter Verlust {large_losses[-1]:.3f}, Testgenauigkeit {large_accuracy:.3f}")

# Die Kurven machen den Unterschied der Batch-Größen sichtbar.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(small_losses, label="Batch-Größe 8")
ax.plot(large_losses, label="Batch-Größe 128")
ax.set_title("Mini-Batches verändern die Verlustkurve")
ax.set_xlabel("Epoche")
ax.set_ylabel("Trainingsverlust")
ax.legend()
plt.show()



### **2.2. SGD mit Momentum**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_02_02.jpg?v=1784807943" width="250">



>* Momentum speichert frühere Update-Richtungen
>* Rauschen sinkt, konsistente Bewegungen werden stärker

>* Momentum glättet unruhige SGD-Schritte
>* Zu viel Schwung kann überschießen

>* Momentum stabilisiert Training trotz zufälliger Mini-Batches
>* Regularisierung und Early Stopping verhindern Overfitting



In [ ]:
#@title Python-Code - SGD mit Momentum

# Dieses Beispiel vergleicht SGD mit Momentum.
# Momentum speichert frühere Update-Richtungen als Geschwindigkeit.
# Die Kurven zeigen stabilere Optimierungsschritte.

import numpy as np
import matplotlib.pyplot as plt

# Wir erzeugen eine kleine, längliche Fehlerlandschaft.
x_values = np.linspace(-4.0, 4.0, 120)
y_values = np.linspace(-2.0, 2.0, 120)

# Das Minimum liegt bei null und ist schmal.
start = np.array([3.5, 1.6])
learning_rate = 0.08
momentum = 0.85
steps = 35

# Diese Funktion liefert den Gradienten der Fehlerlandschaft.
def gradient(point):
    return np.array([0.2 * point[0], 4.0 * point[1]])

# Einfaches SGD nutzt nur den aktuellen Gradienten.
sgd_path = [start.copy()]
point = start.copy()
for step in range(steps):
    point = point - learning_rate * gradient(point)
    sgd_path.append(point.copy())

# Momentum mischt den aktuellen Gradienten mit früherer Bewegung.
momentum_path = [start.copy()]
point = start.copy()
velocity = np.zeros(2)
for step in range(steps):
    velocity = momentum * velocity - learning_rate * gradient(point)
    point = point + velocity
    momentum_path.append(point.copy())

# Wir prüfen, ob beide Pfade gleich lang sind.
if len(sgd_path) != steps + 1 or len(momentum_path) != steps + 1:
    raise ValueError("Die Optimierungspfade haben eine unerwartete Länge.")

sgd_path = np.array(sgd_path)
momentum_path = np.array(momentum_path)

# Die Konturen zeigen gleiche Verlustwerte.
xx, yy = np.meshgrid(x_values, y_values)
loss = 0.1 * xx**2 + 2.0 * yy**2

print("Vergleich nach 35 Schritten:")
print(f"SGD Abstand zum Minimum: {np.linalg.norm(sgd_path[-1]):.3f}")
print(f"Momentum Abstand zum Minimum: {np.linalg.norm(momentum_path[-1]):.3f}")
print(f"Verwendete Lernrate: {learning_rate}, Momentum: {momentum}")

# Ein einzelnes Diagramm macht die Bewegungen vergleichbar.
fig, ax = plt.subplots(figsize=(7, 5))
ax.contour(xx, yy, loss, levels=18, cmap="Greys")
ax.plot(sgd_path[:, 0], sgd_path[:, 1], "o-", label="SGD")
ax.plot(momentum_path[:, 0], momentum_path[:, 1], "o-", label="SGD mit Momentum")
ax.scatter([0], [0], color="black", s=60, label="Minimum")

ax.set_title("SGD mit und ohne Momentum")
ax.set_xlabel("Gewicht 1")
ax.set_ylabel("Gewicht 2")
ax.legend()
plt.show()



### **2.3. Regularisierung gegen Overfitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_02_03.jpg?v=1784807939" width="250">



>* Overfitting bedeutet Auswendiglernen statt Verallgemeinern
>* Regularisierung fördert einfache, robuste Muster

>* Große Gewichte machen Modelle empfindlicher.
>* Regularisierung fördert robuste Entscheidungen auf neuen Daten.

>* Validierung und Early Stopping erkennen Overfitting
>* Regularisierung gehört zur gesamten Trainingsstrategie



In [ ]:
#@title Python-Code - Regularisierung gegen Overfitting

# Dieses Beispiel zeigt Regularisierung gegen Overfitting.
# Wir vergleichen zwei logistische Klassifikatoren.
# Die Validierungsgenauigkeit zeigt den Nutzen.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Wir erzeugen kleine verrauschte Klassifikationsdaten.
features, labels = make_classification(
    n_samples=500,
    n_features=20,
    n_informative=4,
    n_redundant=2,
    flip_y=0.18,
    class_sep=0.8,
    random_state=42,
)

# Diese Prüfung macht die Datenform sichtbar.
if features.shape != (500, 20):
    raise ValueError("Die erzeugten Daten haben eine unerwartete Form.")

# Die Aufteilung trennt Training und Validierung.
X_train, X_valid, y_train, y_valid = train_test_split(
    features,
    labels,
    test_size=0.4,
    stratify=labels,
    random_state=42,
)

# Skalierung wird nur auf Trainingsdaten gelernt.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Kleines C bedeutet stärkere L2-Regularisierung.
models = {
    "schwach regularisiert": LogisticRegression(C=1000.0, max_iter=1000),
    "stark regularisiert": LogisticRegression(C=0.08, max_iter=1000),
}

train_scores = []
valid_scores = []
weight_sizes = []

# Beide Modelle sehen dieselben Daten.
for model_name, model in models.items():
    model.fit(X_train_scaled, y_train)
    train_scores.append(accuracy_score(y_train, model.predict(X_train_scaled)))
    valid_scores.append(accuracy_score(y_valid, model.predict(X_valid_scaled)))
    weight_sizes.append(float(np.linalg.norm(model.coef_)))

print(f"scikit-learn-Version: {sklearn.__version__}")
print("Vergleich: schwache vs. starke L2-Regularisierung")
print(f"Training: {train_scores[0]:.2f} gegen {train_scores[1]:.2f}")
print(f"Validierung: {valid_scores[0]:.2f} gegen {valid_scores[1]:.2f}")
print(f"Gewichtsnorm: {weight_sizes[0]:.2f} gegen {weight_sizes[1]:.2f}")

# Das Diagramm zeigt Anpassung und Verallgemeinerung.
positions = np.arange(len(models))
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(positions - width / 2, train_scores, width, label="Training")
ax.bar(positions + width / 2, valid_scores, width, label="Validierung")
ax.set_xticks(positions, list(models.keys()))

ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Genauigkeit")
ax.set_title("Regularisierung kann Overfitting verringern")
ax.legend()

plt.show()



## **3. CNN Bausteine**

### **3.1. Faltung verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_03_01.jpg?v=1784807949" width="250">



>* Filter erkennen lokale Bildmuster in Ausschnitten
>* Geteilte Filter finden Merkmale überall

>* Geteilte Filter sparen Parameter und erkennen Verschiebungen
>* Schichten bauen einfache zu komplexen Merkmalen

>* Filter erzeugen Merkmalskarten aus lokalen Bereichen
>* Mehrere Kanäle erfassen unterschiedliche Muster



In [ ]:
#@title Python-Code - Faltung verstehen

# Wir berechnen eine kleine Faltung von Hand.
# Der Filter sucht eine einfache vertikale Kante.
# Die Ausgabe zeigt Eingabe, Filter und Merkmalskarte.

import numpy as np
import matplotlib.pyplot as plt

# Dieses synthetische Bild hat links dunkle und rechts helle Pixel.
image = np.array(
    [[0, 0, 0, 5, 5, 5],
     [0, 0, 0, 5, 5, 5],
     [0, 0, 0, 5, 5, 5],
     [0, 0, 0, 5, 5, 5],
     [0, 0, 0, 5, 5, 5],
     [0, 0, 0, 5, 5, 5]],
    dtype=float,
)

# Dieser Filter reagiert stark auf einen Hell-Dunkel-Wechsel.
kernel = np.array(
    [[-1, 0, 1],
     [-1, 0, 1],
     [-1, 0, 1]],
    dtype=float,
)

# Die Ausgabegröße folgt aus Eingabegröße minus Filtergröße plus eins.
output_height = image.shape[0] - kernel.shape[0] + 1
output_width = image.shape[1] - kernel.shape[1] + 1
feature_map = np.zeros((output_height, output_width), dtype=float)

# An jeder Position wird Ausschnitt mal Filter summiert.
for row in range(output_height):
    for col in range(output_width):
        patch = image[row:row + 3, col:col + 3]
        feature_map[row, col] = np.sum(patch * kernel)

# Diese Prüfung macht die geplanten Tensorformen sichtbar.
expected_shape = (4, 4)
if feature_map.shape != expected_shape:
    raise ValueError("Die Merkmalskarte hat nicht die erwartete Form.")

print("Eingabeform: 6x6, Filterform: 3x3, Ausgabeform: 4x4")
print("Stärkste Filterantwort:", int(np.max(feature_map)))
print("Position der stärksten Antwort:", tuple(np.argwhere(feature_map == 15)[0]))

# Eine Achse zeigt alle drei Matrizen als beschriftete Textblöcke.
fig, ax = plt.subplots(figsize=(8, 4))
ax.axis("off")
ax.set_title("Faltung: lokaler Filter erzeugt eine Merkmalskarte")

image_text = "Eingabe\n" + str(image.astype(int))
kernel_text = "Filter\n" + str(kernel.astype(int))
feature_text = "Merkmalskarte\n" + str(feature_map.astype(int))

ax.text(0.02, 0.75, image_text, family="monospace", fontsize=11, va="top")
ax.text(0.42, 0.75, kernel_text, family="monospace", fontsize=11, va="top")
ax.text(0.68, 0.75, feature_text, family="monospace", fontsize=11, va="top")

plt.show()



### **3.2. Padding und Pooling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_03_02.jpg?v=1784807945" width="250">



>* Padding schützt Ränder vor Informationsverlust
>* Tensorgrößen bleiben planbar und stabil

>* Pooling verdichtet Feature Maps räumlich.
>* Max und Average Pooling schaffen Robustheit.

>* Padding bewahrt Formen, Pooling verdichtet gezielt
>* Balance schützt Details, Robustheit und Ressourcen



In [ ]:
#@title Python-Code - Padding und Pooling

# Dieses Beispiel zeigt Padding und Pooling.
# Kleine Matrizen machen Tensorformen sichtbar.
# Die Ausgabe vergleicht Größen und Werte.

import numpy as np

# Diese Eingabe steht für eine kleine Feature Map.
feature_map = np.array(
    [[1, 2, 0, 1], [3, 4, 1, 0], [0, 2, 5, 1], [1, 0, 2, 3]]
)

# Padding fügt außen einen Rahmen aus Nullen hinzu.
padded_map = np.pad(feature_map, pad_width=1, mode="constant", constant_values=0)

# Max-Pooling übernimmt pro Fenster den größten Wert.
def max_pool_2x2(input_map):
    output_height = input_map.shape[0] // 2
    output_width = input_map.shape[1] // 2
    pooled = np.zeros((output_height, output_width), dtype=input_map.dtype)

    for row in range(output_height):
        for col in range(output_width):
            window = input_map[row * 2:row * 2 + 2, col * 2:col * 2 + 2]
            pooled[row, col] = np.max(window)

    return pooled

# Average-Pooling bildet pro Fenster den Mittelwert.
def average_pool_2x2(input_map):
    output_height = input_map.shape[0] // 2
    output_width = input_map.shape[1] // 2
    pooled = np.zeros((output_height, output_width), dtype=float)

    for row in range(output_height):
        for col in range(output_width):
            window = input_map[row * 2:row * 2 + 2, col * 2:col * 2 + 2]
            pooled[row, col] = np.mean(window)

    return pooled

# Die Formen zeigen die Wirkung auf Höhe und Breite.
max_pooled = max_pool_2x2(feature_map)
average_pooled = average_pool_2x2(feature_map)

# Eine einfache Prüfung schützt vor unerwarteten Formen.
if padded_map.shape != (6, 6):
    raise ValueError("Das Padding sollte eine 6x6-Matrix erzeugen.")

# Die Ausgabe bleibt kurz und zeigt nur zentrale Ergebnisse.
print("Originalform:", feature_map.shape)
print("Form nach Padding:", padded_map.shape)
print("Form nach 2x2-Max-Pooling:", max_pooled.shape)
print("Form nach 2x2-Average-Pooling:", average_pooled.shape)
print("Original oben links:", feature_map[:2, :2].tolist())
print("Max-Pooling Ergebnis:", max_pooled.tolist())
print("Average-Pooling Ergebnis:", np.round(average_pooled, 2).tolist())



### **3.3. Tensorformen planen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_14/Lecture_B/image_03_03.jpg?v=1784807947" width="250">



>* Dimensionen durch jede CNN-Schicht verfolgen
>* Passende Formen verhindern spätere Schichtfehler

>* Auflösung und Kanäle getrennt betrachten
>* Pooling verdichtet, Faltungen erzeugen Merkmale

>* Übergänge und Parametergrößen sicher planen
>* Effiziente CNNs bleiben hardwaregerecht und detailbewusst



In [ ]:
#@title Python-Code - Tensorformen planen

# Wir planen Tensorformen in einem kleinen CNN.
# Faltung und Pooling verändern Höhe und Breite.
# Die Ausgabe zeigt jede Formänderung übersichtlich.

import numpy as np

# Diese Funktion berechnet eine räumliche Ausgabegröße.
def output_size(input_size, kernel_size, padding, stride):
    numerator = input_size + 2 * padding - kernel_size
    return numerator // stride + 1

# Wir beschreiben einen kleinen Bildstapel im NHWC-Format.
batch_size = 8
height = 28
width = 28
channels = 1

# Jede Zeile beschreibt eine typische CNN-Operation.
layers = [
    ("Eingabe", 0, 0, 1, channels),
    ("Faltung 3x3, 16 Filter", 3, 1, 1, 16),
    ("Max-Pooling 2x2", 2, 0, 2, 16),
    ("Faltung 3x3, 32 Filter", 3, 1, 1, 32),
    ("Max-Pooling 2x2", 2, 0, 2, 32),
]

# Wir sammeln die Formen nach jeder Schicht.
shape_rows = []
current_height = height
current_width = width
current_channels = channels

# Die Eingabeform wird zuerst notiert.
shape_rows.append((layers[0][0], batch_size, height, width, channels))

# Danach berechnen wir jede weitere Form Schritt für Schritt.
for name, kernel_size, padding, stride, new_channels in layers[1:]:
    current_height = output_size(current_height, kernel_size, padding, stride)
    current_width = output_size(current_width, kernel_size, padding, stride)
    current_channels = new_channels
    shape_rows.append((name, batch_size, current_height, current_width, current_channels))

# Eine einfache Prüfung schützt vor unmöglichen Formen.
if current_height <= 0 or current_width <= 0:
    raise ValueError("Die geplante CNN-Form ist räumlich ungültig.")

# Die flache Größe ist wichtig vor einer Dense-Schicht.
flattened_features = current_height * current_width * current_channels

# Die Ausgabe bleibt kurz und zeigt den Datenfluss.
print("Schicht | Tensorform im Format (N, H, W, C)")
for name, n, h, w, c in shape_rows:
    print(f"{name}: ({n}, {h}, {w}, {c})")

# Diese Zahl bestimmt die Eingabegröße der nächsten Dense-Schicht.
print(f"Werte pro Beispiel nach Flatten: {flattened_features}")



# <font color="#418FDE" size="6.5" uppercase>**MLP und Faltungen**</font>


In this lecture, you learned to:
- Trainieren ein kleines Zwei-Schichten-MLP mit NumPy auf einfachen Klassifikationsdaten. 
- Vergleichen Mini-Batches, Shuffling, SGD, Momentum, Regularisierung und Early Stopping. 
- Implementieren grundlegende Faltungs- und Poolingoperationen und planen Tensorformen. 

In the next Module (Module 15), we will go over 'TensorFlow Keras'